# Phân tích ROI Index cho các task IT

Notebook này trình bày toàn bộ quy trình phân tích kinh tế: kiểm tra dữ liệu đầu vào, xây dựng ROI Index, phân vùng chiến lược, giải thích kết quả, kiểm tra độ nhạy và xuất bảng bàn giao.

> **Lưu ý:** ROI Index là chỉ số ưu tiên kinh tế tương đối, không phải ROI kế toán. WORKBank chưa cung cấp chi phí xây dựng, vận hành và giám sát AI Agent nên chưa thể tính `(lợi ích - chi phí) / chi phí`.

## 1. Thiết lập môi trường và tải dữ liệu
Notebook chỉ đọc **một đầu vào** là `data/processed/it_master.csv`. Sau đó notebook gọi `build_roi_table(master)` để tính toàn bộ ROI trong bộ nhớ. File `roi_index.csv` không cần tồn tại trước khi chạy và chỉ được tạo ở phần xuất kết quả cuối notebook.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.roi_index import build_roi_table

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

INPUT_PATH = ROOT / 'data' / 'processed' / 'it_master.csv'
SCREENSHOT_DIR = ROOT / 'docs' / 'screenshots'
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

def save_chart(fig, filename, width=1200, height=700):
    output_path = SCREENSHOT_DIR / filename
    fig.write_image(str(output_path), width=width, height=height, scale=2)
    print(f'Đã lưu biểu đồ: {output_path}')
    return output_path

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Chưa có dữ liệu đầu vào: {INPUT_PATH}. Hãy chạy src.data_processing trước.')

# INPUT duy nhất: bảng master do Thành viên 1 bàn giao.
master = pd.read_csv(INPUT_PATH)
# Tính mới ROI trong bộ nhớ; KHÔNG đọc data/processed/roi_index.csv.
roi = build_roi_table(master)
print(f'Đầu vào đã đọc: {INPUT_PATH}')
print(f'Dữ liệu master: {master.shape[0]} task × {master.shape[1]} cột')
print(f'Bảng ROI vừa tính trong bộ nhớ: {roi.shape[0]} task × {roi.shape[1]} cột')
print('Notebook chưa đọc hoặc ghi roi_index.csv ở bước này.')

Đầu vào đã đọc: d:\Python\DAT713_AI_Agent_Project\AI_Agent_Project\data\processed\it_master.csv
Dữ liệu master: 186 task × 49 cột
Bảng ROI vừa tính trong bộ nhớ: 186 task × 23 cột
Notebook chưa đọc hoặc ghi roi_index.csv ở bước này.


## 2. Kiểm tra chất lượng dữ liệu đầu vào
Năm biến bắt buộc gồm lương, số lao động, tần suất, tầm quan trọng và Automation Capacity. Task thiếu bất kỳ thành phần nào sẽ được giữ lại nhưng gắn nhãn **Chưa đủ dữ liệu**, không tự động điền bằng 0.

In [2]:
required = [
    'Occupation Mean Annual Wage',
    'Occupation Employment',
    'Frequency',
    'Importance',
    'Expert_Automation Capacity Rating',
    'Worker_Time',
]
quality = pd.DataFrame({
    'Kiểu dữ liệu': master[required].dtypes.astype(str),
    'Số thiếu': master[required].isna().sum(),
    'Tỷ lệ thiếu (%)': master[required].isna().mean().mul(100),
    'Nhỏ nhất': master[required].min(numeric_only=True),
    'Trung vị': master[required].median(numeric_only=True),
    'Lớn nhất': master[required].max(numeric_only=True),
})
display(quality)
print(f"Task đủ dữ liệu ROI: {int(roi['ROI Data Complete'].sum())}/{len(roi)}")

,Kiểu dữ liệu,Số thiếu,Tỷ lệ thiếu (%),Nhỏ nhất,Trung vị,Lớn nhất
Occupation Mean Annual Wage,float64,63,33.871,"64,990.000","107,440.000","187,990.000"
Occupation Employment,float64,63,33.871,"38,480.000","179,430.000","697,210.000"
Frequency,float64,0,0.000,3.000,4.000,7.000
Importance,float64,0,0.000,2.770,3.895,4.790
Expert_Automation Capacity Rating,float64,55,29.570,1.500,3.500,5.000
Worker_Time,float64,55,29.570,1.200,2.286,3.500


Task đủ dữ liệu ROI: 102/186


## 3. Phương pháp tính ROI Index

### 3.1. Quy mô giá trị lao động
`Market Scale = percentile-rank(log(1 + Annual Wage × Employment))` theo các nghề duy nhất

Biến đổi log hạn chế việc nghề có số lao động rất lớn áp đảo hoàn toàn bảng xếp hạng.

### 3.2. Mức độ hiện diện của task trong công việc
`Time Share Proxy = 0.10 + 0.90 × (Worker Time − 1) / 4`

`Frequency Intensity = (Frequency − 1) / 6`

`Importance Intensity = (Importance − 1) / 4`

`Task Exposure = 0.50 × Time Share Proxy + 0.25 × Frequency Intensity + 0.25 × Importance Intensity`

### 3.3. Khả năng tự động hóa
`Automation Potential = (Expert Automation Capacity − 1) / 4`

### 3.4. Điểm tổng hợp
`Economic Potential Raw = Market Scale × Task Exposure × Automation Potential`

`ROI Index = MinMax(Economic Potential Raw)`

Các thang Frequency, Importance và Automation được chuẩn hóa theo neo lý thuyết gốc, không theo min/max quan sát. Codebook xác định Worker Time từ 1 = 10% đến 5 = 100% thời gian làm việc; các mức giữa được nội suy tuyến tính và chỉ được gọi là proxy, không phải số giờ. Phép nhân bảo đảm một thành phần rất cao không thể che lấp hoàn toàn một thành phần gần bằng 0.

In [3]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

complete = roi[roi['ROI Data Complete']].copy()

components = ['Market Scale','Task Exposure','Automation Potential','ROI Index']

subplot_titles = ['Quy mô thị trường','Mức độ tiếp xúc với tác vụ','Tiềm năng tự động hóa','Chỉ số ROI']

fig = make_subplots(rows=2, cols=2, subplot_titles=subplot_titles)

positions = [(1,1), (1,2), (2,1), (2,2)]
colors = ['royalblue', 'tomato', 'mediumseagreen', 'mediumpurple']

for column, title, color, (r, c) in zip(components, subplot_titles, colors, positions):
    fig.add_trace(
        go.Histogram(
            x=complete[column],
            nbinsx=15,
            marker=dict(color=color, line=dict(color='white',width=1)), showlegend=False), row=r, col=c)


for annotation in fig.layout.annotations:
    annotation.text = f"<b>{annotation.text}</b>"
    annotation.font = dict(size=17)

fig.update_layout(
    template='plotly_white',

    title=dict(text="<b>Phân bố các thành phần của chỉ số ROI</b>", x=0.5, xanchor="center", font=dict(size=22)),
    height=700,
    width=1100,
    bargap=0.08
)

fig.update_xaxes(title_text="Giá trị")
fig.update_yaxes(title_text="Tần suất")

#save_chart(fig, 'Phân phối các thành phần_ROI_Index.png', height=750)

fig.show()

## 4. Phân vùng chiến lược
Ngưỡng được xác định từ phân phối của các task đủ dữ liệu: từ phân vị 75% là **Tự động hóa ngay**, phân vị 40%–75% là **Cân nhắc**, dưới phân vị 40% là **Giữ nguyên/Theo dõi**. Cách này thích ứng với dữ liệu thực tế hơn ngưỡng cố định tùy ý.

In [4]:
valid_roi = roi.loc[roi["ROI Data Complete"], "ROI Index"].dropna()

q40 = valid_roi.quantile(0.40)
q75 = valid_roi.quantile(0.75)

print(f"Q40 = {q40:.6f}")
print(f"Q75 = {q75:.6f}")

Q40 = 0.252359
Q75 = 0.477667


In [5]:
zone_order = ['Tự động hóa ngay', 'Cân nhắc', 'Giữ nguyên / Theo dõi', 'Chưa đủ dữ liệu']
zone_colors = {
    'Tự động hóa ngay': '#16a34a',
    'Cân nhắc': '#f59e0b',
    'Giữ nguyên / Theo dõi': '#64748b',
    'Chưa đủ dữ liệu': '#cbd5e1',
}
zone_counts = roi['Strategy Zone'].value_counts().reindex(zone_order, fill_value=0).reset_index()
zone_counts.columns = ['Vùng chiến lược', 'Số task']
display(zone_counts)
fig = px.bar(zone_counts, x='Vùng chiến lược', y='Số task', color='Vùng chiến lược',
             color_discrete_map=zone_colors, text='Số task', title='Số task theo vùng chiến lược')
fig.update_layout(
    showlegend=False,
    title=dict(text="<b>Số task theo vùng chiến lược</b>", x=0.5, xanchor="center", font=dict(size=22)), template="plotly_white")
# save_chart(fig, 'Số task theo vùng chiến lược.png')
fig.show()

,Vùng chiến lược,Số task
0,Tự động hóa ngay,26
1,Cân nhắc,35
2,Giữ nguyên / Theo dõi,41
3,Chưa đủ dữ liệu,84


## 5. Bảng xếp hạng task theo ROI Index
Ngoài điểm ROI, cần xem `Data Confidence`: mức **Cao** yêu cầu đủ dữ liệu, có 3 chuyên gia và ít nhất 8 worker đánh giá; các task đủ dữ liệu còn lại được xếp **Trung bình**. Đây là quy tắc mô tả sức mạnh bằng chứng, không phải khoảng tin cậy thống kê.

In [6]:
top_columns = [
    'Task ID', 'Occupation (O*NET-SOC Title)', 'Task', 'ROI Index',
    'Market Scale', 'Task Exposure', 'Time Share Proxy', 'Automation Potential',
    'Data Confidence', 'Strategy Zone',
]
top10 = complete.nlargest(10, 'ROI Index')[top_columns]
display(top10.style.format({column: '{:.3f}' for column in ['ROI Index', 'Market Scale', 'Task Exposure', 'Time Share Proxy', 'Automation Potential']}))

plot_top10 = top10.sort_values('ROI Index')
fig = px.bar(plot_top10, x='ROI Index', y='Task', color='Data Confidence', orientation='h',
             hover_data=['Occupation (O*NET-SOC Title)'], title='Top 10 task theo ROI Index')
fig.update_layout(margin=dict(l=450,r=30,t=80,b=50))
# save_chart(fig, 'Top 10 task theo ROI Index.png', height=750)
fig.show()

,Task ID,Occupation (O*NET-SOC Title),Task,ROI Index,Market Scale,Task Exposure,Time Share Proxy,Automation Potential,Data Confidence,Strategy Zone
0,1282,Computer User Support Specialists,Answer user inquiries regarding computer software or hardware operation to resolve problems.,1.000,0.818,0.654,0.460,0.875,Trung bình,Tự động hóa ngay
1,1285,Computer User Support Specialists,Oversee the daily performance of computer systems.,0.949,0.818,0.544,0.381,1.000,Trung bình,Tự động hóa ngay
2,1287,Computer User Support Specialists,"Maintain records of daily data communication transactions, problems and remedial actions taken, or installation activities.",0.862,0.818,0.540,0.438,0.917,Trung bình,Tự động hóa ngay
3,966,Computer and Information Systems Managers,"Manage backup, security and user help systems.",0.837,1.000,0.630,0.527,0.625,Trung bình,Tự động hóa ngay
4,1283,Computer User Support Specialists,Enter commands and observe system functioning to verify correct operations and detect errors.,0.832,0.818,0.522,0.389,0.917,Trung bình,Tự động hóa ngay
5,15206,Network and Computer Systems Administrators,"Configure, monitor, and maintain email applications or virus protection software.",0.795,0.727,0.515,0.325,1.000,Trung bình,Tự động hóa ngay
6,3466,Computer Systems Analysts,"Use object-oriented programming languages, as well as client and server applications development processes and multimedia and Internet technology.",0.782,0.909,0.540,0.325,0.750,Cao,Tự động hóa ngay
7,14648,Software Quality Assurance Analysts and Testers,Document test procedures to ensure replicability and compliance with standards.,0.776,0.545,0.670,0.595,1.000,Trung bình,Tự động hóa ngay
8,3478,Computer Systems Analysts,"Review and analyze computer printouts and performance indicators to locate code problems, and correct errors by correcting codes.",0.774,0.909,0.458,0.425,0.875,Trung bình,Tự động hóa ngay
9,14644,Software Quality Assurance Analysts and Testers,Create or maintain databases of known test defects.,0.738,0.545,0.639,0.550,1.000,Trung bình,Tự động hóa ngay


## 6. Giải thích vì sao task có ROI cao
Biểu đồ dưới đây phân rã hai động lực chính. Điểm ở góc trên bên phải vừa có quy mô kinh tế lớn vừa có khả năng tự động hóa cao; kích thước điểm thể hiện ROI Index, còn `Task Exposure` xuất hiện khi rê chuột.

In [7]:
fig = px.scatter(
    complete, x='Market Scale', y='Automation Potential', size='ROI Index',
    color='Strategy Zone', color_discrete_map=zone_colors, hover_name='Task',
    hover_data={'Occupation (O*NET-SOC Title)': True, 'Task Exposure': ':.3f',
                'ROI Index': ':.3f', 'Data Confidence': True},
    size_max=28, title='Quy mô kinh tế và khả năng tự động hóa theo task',
)
save_chart(fig, 'Quy mô kinh tế và khả năng tự động hóa theo task.png')
fig.show()

Đã lưu biểu đồ: d:\Python\DAT713_AI_Agent_Project\AI_Agent_Project\docs\screenshots\Quy mô kinh tế và khả năng tự động hóa theo task.png


## 7. Tương quan từng thành phần với ROI Index
Spearman correlation được dùng vì ROI Index là chỉ số xếp hạng tương đối và được tạo bởi mô hình nhân, nên không giả định quan hệ tuyến tính. Kết quả cho biết thành phần nào có quan hệ thứ hạng mạnh hơn với ROI Index; đây không phải hệ số nhân quả hoặc phần trăm đóng góp.

In [8]:
component_columns = [
    'Market Scale',
    'Task Exposure',
    'Automation Potential',
    'Time Share Proxy',
    'Frequency Intensity',
    'Importance Intensity',
]
component_correlations = (
    complete[['ROI Index', *component_columns]]
    .corr(method='spearman')['ROI Index']
    .drop('ROI Index')
    .rename('Spearman ρ với ROI Index')
    .rename_axis('Thành phần')
    .reset_index()
)
display(component_correlations.style.format({'Spearman ρ với ROI Index': '{:.3f}'}))

fig = px.bar(
    component_correlations.sort_values('Spearman ρ với ROI Index'),
    x='Spearman ρ với ROI Index',
    y='Thành phần',
    orientation='h',
    text='Spearman ρ với ROI Index',
    title='Mức liên hệ thứ hạng giữa từng thành phần và ROI Index',
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(xaxis_range=[0, 0.75], yaxis_title=None)
save_chart(fig, 'Tương_quan_các_thành_phần_với_ROI_Index.png')
fig.show()

print('Diễn giải: Market Scale có tương quan mạnh nhất; ROI Index không phải thước đo năng lực AI thuần túy.')

,Thành phần,Spearman ρ với ROI Index
0,Market Scale,0.694
1,Task Exposure,0.397
2,Automation Potential,0.399
3,Time Share Proxy,0.265
4,Frequency Intensity,0.315
5,Importance Intensity,0.197


Đã lưu biểu đồ: d:\Python\DAT713_AI_Agent_Project\AI_Agent_Project\docs\screenshots\Tương_quan_các_thành_phần_với_ROI_Index.png


Diễn giải: Market Scale có tương quan mạnh nhất; ROI Index không phải thước đo năng lực AI thuần túy.


## 8. Tổng hợp theo nghề
ROI ở cấp nghề được mô tả bằng trung vị thay vì tổng, tránh nghề có nhiều task mặc nhiên đứng đầu. Bảng đồng thời cho biết số task đủ dữ liệu và số task thuộc vùng ưu tiên.

In [9]:
occupation_summary = (
    complete.groupby('Occupation (O*NET-SOC Title)')
    .agg(
        Task_Count=('Task ID', 'count'),
        Median_ROI=('ROI Index', 'median'),
        Mean_ROI=('ROI Index', 'mean'),
        Max_ROI=('ROI Index', 'max'),
        Immediate_Tasks=('Strategy Zone', lambda values: (values == 'Tự động hóa ngay').sum()),
    )
    .sort_values(['Median_ROI', 'Max_ROI'], ascending=False)
    .reset_index()
)
display(occupation_summary)
fig = px.bar(occupation_summary.head(12).sort_values('Median_ROI'), x='Median_ROI',
             y='Occupation (O*NET-SOC Title)', orientation='h', color='Immediate_Tasks',
             title='Top nghề theo trung vị ROI Index')
fig.update_layout(yaxis_title=None)
save_chart(fig, 'Top nghề theo trung vị ROI Index.png', height=750)
fig.show()

,Occupation (O*NET-SOC Title),Task_Count,Median_ROI,Mean_ROI,Max_ROI,Immediate_Tasks
0,Computer User Support Specialists,7,0.832,0.684,1.000,5
1,Computer Systems Analysts,9,0.592,0.572,0.782,8
2,Network and Computer Systems Administrators,10,0.436,0.416,0.795,3
3,Software Quality Assurance Analysts and Testers,17,0.433,0.468,0.776,6
4,Information Security Analysts,5,0.388,0.356,0.410,0
5,Computer Network Support Specialists,9,0.349,0.315,0.481,1
6,Computer and Information Systems Managers,9,0.334,0.429,0.837,3
7,Computer Programmers,11,0.266,0.257,0.399,0
8,Database Administrators,6,0.156,0.157,0.233,0
9,Web Developers,13,0.127,0.128,0.240,0


Đã lưu biểu đồ: d:\Python\DAT713_AI_Agent_Project\AI_Agent_Project\docs\screenshots\Top nghề theo trung vị ROI Index.png


## 9. Kiểm tra độ nhạy của bảng xếp hạng

### 9.1. Thay đổi trọng số bên trong Task Exposure

Mô hình cơ sở sử dụng trọng số 50% cho Time Share Proxy, 25% cho Frequency Intensity và 25% cho Importance Intensity.

Các kịch bản dưới đây kiểm tra mức độ ổn định của thứ hạng khi thay đổi hợp lý các trọng số này. Spearman càng gần 1 và số task Top 10 được giữ lại càng lớn thì kết quả càng ổn định.

In [10]:
# Tập task đủ dữ liệu ROI
complete = roi.loc[roi["ROI Data Complete"]].copy()

# Top 10 của mô hình ROI cơ sở
base_top10_ids = set(
    complete.nlargest(10, "ROI Index")["Task ID"]
)

# Các trọng số lần lượt là:
# Time Share Proxy, Frequency Intensity, Importance Intensity
exposure_scenarios = {
    "Cân bằng hơn (40%-30%-30%)": (0.40, 0.30, 0.30),
    "Mô hình cơ sở (50%-25%-25%)": (0.50, 0.25, 0.25),
    "Nhấn mạnh thời gian (60%-20%-20%)": (0.60, 0.20, 0.20),
    "Nhấn mạnh tần suất (40%-40%-20%)": (0.40, 0.40, 0.20),
    "Nhấn mạnh tầm quan trọng (40%-20%-40%)": (0.40, 0.20, 0.40),
}

sensitivity_results = []
scenario_scores = complete[
    ["Task ID", "Occupation (O*NET-SOC Title)", "Task", "ROI Index"]
].copy()

base_rank = complete["ROI Index"].rank(
    method="average",
    ascending=False,
)

for scenario_name, (
    time_weight,
    frequency_weight,
    importance_weight,
) in exposure_scenarios.items():

    # Tính lại Task Exposure theo trọng số của từng kịch bản
    scenario_exposure = (
        time_weight * complete["Time Share Proxy"]
        + frequency_weight * complete["Frequency Intensity"]
        + importance_weight * complete["Importance Intensity"]
    )

    # Giữ nguyên cấu trúc mô hình nhân
    scenario_raw = (
        complete["Market Scale"]
        * scenario_exposure
        * complete["Automation Potential"]
    )

    scenario_scores[scenario_name] = scenario_raw

    # Tương quan Spearman được tính từ thứ hạng
    scenario_rank = scenario_raw.rank(
        method="average",
        ascending=False,
    )
    spearman_rho = base_rank.corr(scenario_rank)

    scenario_top10_ids = set(
        complete.assign(Scenario_Score=scenario_raw)
        .nlargest(10, "Scenario_Score")["Task ID"]
    )

    top10_overlap = len(
        base_top10_ids.intersection(scenario_top10_ids)
    )

    sensitivity_results.append({
        "Kịch bản": scenario_name,
        "Time": time_weight,
        "Frequency": frequency_weight,
        "Importance": importance_weight,
        "Spearman ρ": spearman_rho,
        "Top 10 giữ lại": f"{top10_overlap}/10",
    })

exposure_sensitivity = pd.DataFrame(sensitivity_results)

display(
    exposure_sensitivity.style.format({
        "Time": "{:.0%}",
        "Frequency": "{:.0%}",
        "Importance": "{:.0%}",
        "Spearman ρ": "{:.3f}",
    })
)

,Kịch bản,Time,Frequency,Importance,Spearman ρ,Top 10 giữ lại
0,Cân bằng hơn (40%-30%-30%),40%,30%,30%,0.999,10/10
1,Mô hình cơ sở (50%-25%-25%),50%,25%,25%,1.000,10/10
2,Nhấn mạnh thời gian (60%-20%-20%),60%,20%,20%,0.998,10/10
3,Nhấn mạnh tần suất (40%-40%-20%),40%,40%,20%,0.996,9/10
4,Nhấn mạnh tầm quan trọng (40%-20%-40%),40%,20%,40%,0.999,10/10


In [15]:
sensitivity_plot = exposure_sensitivity.copy()
sensitivity_plot["Top 10 giữ lại (số task)"] = (
    sensitivity_plot["Top 10 giữ lại"]
    .str.split("/")
    .str[0]
    .astype(int)
)

fig = px.bar(
    sensitivity_plot,
    x="Kịch bản",
    y="Spearman ρ",
    color="Top 10 giữ lại (số task)",
    text="Spearman ρ",
    title="Độ ổn định của ROI Index khi thay đổi trọng số Task Exposure",
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    yaxis_range=[0.99, 1.002],
    xaxis_title=None,
    title=dict(
        x=0.5,
        xanchor="center",
    ),
)
save_chart(fig, "Độ ổn định của ROI Index khi thay đổi trọng số Task Exposure.png", height=700)
fig.show()

Đã lưu biểu đồ: d:\Python\DAT713_AI_Agent_Project\AI_Agent_Project\docs\screenshots\Độ ổn định của ROI Index khi thay đổi trọng số Task Exposure.png


### 9.2. Kiểm tra mở rộng – thay đổi dạng hàm tổng hợp

Ngoài việc thay đổi trọng số bên trong Task Exposure, nghiên cứu so sánh mô hình nhân cơ sở với ba mô hình cộng có trọng số.

Phân tích này kiểm tra mức độ phụ thuộc của kết quả vào giả định rằng Market Scale, Task Exposure và Automation Potential không thể bù trừ hoàn toàn cho nhau.

In [18]:
scenarios = {
    'Cân bằng': (0.35, 0.30, 0.35),
    'Ưu tiên quy mô kinh tế': (0.50, 0.20, 0.30),
    'Ưu tiên khả thi AI': (0.25, 0.25, 0.50),
}
sensitivity = complete[['Task ID', 'Occupation (O*NET-SOC Title)', 'Task', 'ROI Index']].copy()
for name, (market_w, task_w, automation_w) in scenarios.items():
    sensitivity[name] = (
        market_w * complete['Market Scale']
        + task_w * complete['Task Exposure']
        + automation_w * complete['Automation Potential']
    )

rank_columns = ['ROI Index', *scenarios]
rank_correlation = sensitivity[rank_columns].rank(ascending=False).corr(method='spearman')
display(rank_correlation.style.format('{:.3f}'))

top_sets = {column: set(sensitivity.nlargest(10, column)['Task ID']) for column in rank_columns}
stable_top_tasks = set.intersection(*top_sets.values())
print(f'Số task xuất hiện trong Top 10 của tất cả kịch bản: {len(stable_top_tasks)}')
display(sensitivity[sensitivity['Task ID'].isin(stable_top_tasks)].sort_values('ROI Index', ascending=False))

,ROI Index,Cân bằng,Ưu tiên quy mô kinh tế,Ưu tiên khả thi AI
ROI Index,1.000,0.963,0.919,0.830
Cân bằng,0.963,1.000,0.947,0.879
Ưu tiên quy mô kinh tế,0.919,0.947,1.000,0.713
Ưu tiên khả thi AI,0.830,0.879,0.713,1.000


Số task xuất hiện trong Top 10 của tất cả kịch bản: 6


,Task ID,Occupation (O*NET-SOC Title),Task,ROI Index,Cân bằng,Ưu tiên quy mô kinh tế,Ưu tiên khả thi AI
0,1282,Computer User Support Specialists,Answer user inquiries regarding computer software or hardware operation to resolve problems.,1.000,0.789,0.802,0.806
1,1285,Computer User Support Specialists,Oversee the daily performance of computer systems.,0.949,0.800,0.818,0.841
2,1287,Computer User Support Specialists,"Maintain records of daily data communication transactions, problems and remedial actions taken, ...",0.862,0.769,0.792,0.798
4,1283,Computer User Support Specialists,Enter commands and observe system functioning to verify correct operations and detect errors.,0.832,0.764,0.788,0.793
5,15206,Network and Computer Systems Administrators,"Configure, monitor, and maintain email applications or virus protection software.",0.795,0.759,0.767,0.811
8,3478,Computer Systems Analysts,"Review and analyze computer printouts and performance indicators to locate code problems, and co...",0.774,0.762,0.809,0.779


## 10. Phân tích dữ liệu chưa đầy đủ
Không nên bỏ qua 84 task chưa đủ dữ liệu. Bảng sau xác định chính xác thành phần bị thiếu để nhóm có thể bổ sung dữ liệu hoặc ghi giới hạn trong báo cáo.

In [12]:
missing_detail = master[['Task ID', 'Occupation (O*NET-SOC Title)', 'Task', *required]].copy()
missing_detail['Thiếu biến'] = missing_detail[required].apply(
    lambda row: ', '.join(row.index[row.isna()]), axis=1
)
missing_detail = missing_detail[missing_detail['Thiếu biến'] != '']
display(missing_detail['Thiếu biến'].value_counts().rename_axis('Nhóm biến thiếu').reset_index(name='Số task'))
display(missing_detail[['Task ID', 'Occupation (O*NET-SOC Title)', 'Task', 'Thiếu biến']].head(20))

,Nhóm biến thiếu,Số task
0,"Occupation Mean Annual Wage, Occupation Employment, Expert_Automation Capacity Rating, Worker_Time",34
1,"Occupation Mean Annual Wage, Occupation Employment",29
2,"Expert_Automation Capacity Rating, Worker_Time",21


,Task ID,Occupation (O*NET-SOC Title),Task,Thiếu biến
0,18965,Computer Network Architects,Develop and implement solutions for network problems.,"Expert_Automation Capacity Rating, Worker_Time"
1,18974,Computer Network Architects,Estimate time and materials needed to complete projects.,"Expert_Automation Capacity Rating, Worker_Time"
2,18977,Computer Network Architects,"Maintain networks by performing activities such as file addition, deletion, or backup.","Expert_Automation Capacity Rating, Worker_Time"
3,18979,Computer Network Architects,"Monitor and analyze network performance and reports on data input or output to detect problems, ...","Expert_Automation Capacity Rating, Worker_Time"
8,18994,Computer Network Support Specialists,Evaluate local area network (LAN) or wide area network (WAN) performance data to ensure sufficie...,"Expert_Automation Capacity Rating, Worker_Time"
9,19001,Computer Network Support Specialists,"Test computer software or hardware, using standard diagnostic testing equipment and procedures.","Expert_Automation Capacity Rating, Worker_Time"
13,19006,Computer Network Support Specialists,Create or update technical documentation for network installations or changes to existing instal...,"Expert_Automation Capacity Rating, Worker_Time"
14,19007,Computer Network Support Specialists,Document help desk requests and resolutions.,"Expert_Automation Capacity Rating, Worker_Time"
15,19008,Computer Network Support Specialists,Maintain logs of network activity.,"Expert_Automation Capacity Rating, Worker_Time"
16,19009,Computer Network Support Specialists,"Monitor industry Web sites or publications for information about patches, releases, viruses, or ...","Expert_Automation Capacity Rating, Worker_Time"


## 11. Xuất kết quả 
File đầu ra giữ đủ 186 task, bao gồm các thành phần giải thích, trạng thái dữ liệu, độ tin cậy và vùng chiến lược để Thành viên 4 có thể tích hợp sau.

In [13]:
OUTPUT_PATH = ROOT / 'outputs' / 'analysis' / 'roi_index.csv'

# OUTPUT: chỉ ghi sau khi toàn bộ bước kiểm tra và phân tích đã hoàn tất.
roi.to_csv(OUTPUT_PATH, index=False, float_format='%.6f')
print(f'Đã xuất {len(roi)} task ra: {OUTPUT_PATH}')
print('Kiểm tra cuối:')
print('- Task ID duy nhất:', roi['Task ID'].is_unique)
print('- ROI thuộc [0, 1]:', roi.loc[roi['ROI Data Complete'], 'ROI Index'].between(0, 1).all())
print('- Task thiếu dữ liệu không bị gán ROI:', roi.loc[~roi['ROI Data Complete'], 'ROI Index'].isna().all())

Đã xuất 186 task ra: d:\Python\DAT713_AI_Agent_Project\AI_Agent_Project\outputs\analysis\roi_index.csv
Kiểm tra cuối:
- Task ID duy nhất: True
- ROI thuộc [0, 1]: True
- Task thiếu dữ liệu không bị gán ROI: True


## 12. Kết luận và giới hạn

- ROI Index hỗ trợ **sàng lọc ưu tiên**, chưa chứng minh lợi nhuận tài chính thực tế.
- Task thuộc nhóm `Tự động hóa ngay` nên được đối chiếu thêm với Friction Score trước khi triển khai.
- Task ROI cao nhưng độ tin cậy trung bình cần thêm đánh giá chuyên gia hoặc pilot quy mô nhỏ.
- Wage và Employment là dữ liệu cấp nghề, vì vậy các task trong cùng nghề dùng chung Market Scale.
- `Worker_Time` được dùng như tỷ trọng thời gian proxy theo neo 10%–100%; chưa nên công bố số giờ hoặc số tiền tiết kiệm khi chưa có giờ làm thực tế, Adoption Rate và chi phí triển khai.
- Sau pilot, ROI tài chính thực cần tính từ thời gian xử lý, tỷ lệ lỗi, sản lượng, chi phí AI/API/hạ tầng, đào tạo và kiểm duyệt con người.